In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/thanhluanchimto/source-code"
)

METADATA_ROOT = Path(
    "/kaggle/input/datasets/thanhluanchimto/meta-data/metadata"
)

DATA_ROOT = Path(
    "/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing/CelebA_Spoof_/CelebA_Spoof"
)

CHECKPOINT_ROOT = (
    Path("/kaggle/working")
    / "checkpoints"
)

CHECKPOINT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

sys.path.append(
    str(PROJECT_ROOT)
)

print(PROJECT_ROOT.exists())
print(METADATA_ROOT.exists())
print(DATA_ROOT.exists())

True
True
True


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import time

from torch.utils.data import (DataLoader, random_split)
from torchvision import transforms
from tqdm import tqdm

from src.datasets.celeba_spoof_dataset import (CelebASpoofDataset)
from src.models.fsfm_lite import (FSFMLite)

In [3]:
train_df = pd.read_csv(
   METADATA_ROOT
   / "train_df.csv"
)

test_df = pd.read_csv(
   METADATA_ROOT
   / "test_df.csv"
)
DATA_ROOT = Path(
    "/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing/CelebA_Spoof_/CelebA_Spoof"
)

for df in [train_df, test_df]:

    df["image_path"] = df["image_rel_path"].apply(
        lambda x: str(DATA_ROOT / x)
    )

    df["bb_path"] = df["image_rel_path"].apply(
        lambda x: str(
            DATA_ROOT
            / x.replace(".jpg", "_BB.txt")
               .replace(".png", "_BB.txt")
        )
    )

print(train_df.iloc[0]["image_path"])
print(train_df.iloc[0]["bb_path"])

print(train_df.shape)
print(test_df.shape)

/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing/CelebA_Spoof_/CelebA_Spoof/Data/train/4980/spoof/000003.jpg
/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing/CelebA_Spoof_/CelebA_Spoof/Data/train/4980/spoof/000003_BB.txt
(244274, 4)
(25758, 4)


In [4]:
face_transform = transforms.Compose([
   transforms.ToPILImage(),
   transforms.Resize((224,224)),
   transforms.ToTensor(),
   transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

In [5]:
dataset = CelebASpoofDataset(
    train_df,
    transform=face_transform
)
print(len(dataset))

244274


In [6]:
train_size = int(0.9 * len(dataset))
val_size = (len(dataset) - train_size)

train_dataset, val_dataset = random_split(
   dataset,
   [
      train_size,
      val_size
   ]
)

print(len(train_dataset))
print(len(val_dataset))

219846
24428


In [7]:
BATCH_SIZE = 16
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [8]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [9]:
model = FSFMLite()
model = model.to(device)

print("Model Loaded")

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitb14_pretrain.pth


100%|██████████| 330M/330M [00:01<00:00, 333MB/s]


Model Loaded


In [10]:
for p in model.backbone.parameters():
   p.requires_grad = False

trainable_params = sum(
   p.numel()
   for p in model.parameters()
   if p.requires_grad
)

print(f"{trainable_params:,}")

11,420,930


In [11]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Params: {total_params:,}")
print(f"Trainable Params: {trainable_params:,}")

Total Params: 98,001,410
Trainable Params: 11,420,930


In [12]:
criterion = nn.CrossEntropyLoss()

In [13]:
optimizer = torch.optim.AdamW(
   filter(lambda p: p.requires_grad, model.parameters()),
   lr=1e-4,
   weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

In [14]:
def train_one_epoch(model, loader, optimizer, criterion, device):

   model.train()

   total_loss = 0
   total_correct = 0
   total_samples = 0

   pbar = tqdm(loader)

   for batch in pbar:

      images = batch["image"].to(device)
      labels = batch["label"].to(device)

      optimizer.zero_grad()
      logits = model(images)
      loss = criterion(logits, labels)
      pbar.set_postfix(loss=f"{loss.item():.4f}")

      loss.backward()
      torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
      optimizer.step()
      
      preds = logits.argmax(dim=1)
      total_correct += (preds == labels).sum().item()
      total_samples += labels.size(0)
      total_loss += loss.item()

   return (total_loss / len(loader), total_correct / total_samples)

In [15]:
def validate(model, loader, criterion, device):

   model.eval()

   running_loss = 0
   running_correct = 0
   running_total = 0

   with torch.no_grad():

      for batch in loader:
         images = batch["image"].to(device)
         labels = batch["label"].to(device)

         logits = model(images)
         loss = criterion(logits, labels)
         preds = logits.argmax(dim=1)
         running_loss += loss.item()
         running_correct += (preds == labels).sum().item()
         running_total += labels.size(0)

   epoch_loss = (running_loss / len(loader))
   epoch_acc = (running_correct / running_total)
   
   return epoch_loss, epoch_acc

In [16]:
dataset = CelebASpoofDataset(
    train_df,
    transform=face_transform
)

sample = dataset[0]

print(sample["image"].shape)
print(sample["label"])

torch.Size([3, 224, 224])
tensor(1)


In [17]:
batch = next(iter(train_loader))

print(batch["image"].shape)
print(batch["label"].shape)

torch.Size([16, 3, 224, 224])
torch.Size([16])


In [18]:
EPOCHS = 3
best_acc = 0

start_time = time.time()
for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    val_loss, val_acc = validate(
        model,
        val_loader,
        criterion,
        device
    )
    
    scheduler.step(val_acc)
    current_lr = optimizer.param_groups[0]["lr"]
    print(f"LR: {current_lr:.6f}")

    print(f"\nEpoch [{epoch+1}/{EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Acc : {train_acc:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Val Acc    : {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_acc": best_acc
            },
            CHECKPOINT_ROOT
            / "best_fsfm_lite.pth"
        )

        print("Best Model Saved!")

print(f"Training Time: {(time.time()-start_time)/60:.2f} min")

100%|██████████| 13741/13741 [1:14:01<00:00,  3.09it/s, loss=0.0003]


LR: 0.000100

Epoch [1/3]
Train Loss: 0.0452
Train Acc : 0.9847
Val Loss   : 0.0244
Val Acc    : 0.9918
Best Model Saved!


100%|██████████| 13741/13741 [1:13:53<00:00,  3.10it/s, loss=0.0002]


LR: 0.000100

Epoch [2/3]
Train Loss: 0.0229
Train Acc : 0.9923
Val Loss   : 0.0228
Val Acc    : 0.9924
Best Model Saved!


100%|██████████| 13741/13741 [1:13:52<00:00,  3.10it/s, loss=0.0000]


LR: 0.000100

Epoch [3/3]
Train Loss: 0.0162
Train Acc : 0.9950
Val Loss   : 0.0227
Val Acc    : 0.9936
Best Model Saved!
Training Time: 242.38 min
